> **ARCHIVED research notebook** - kept for provenance and understanding only.
> It is **superseded by the `cloaq` package** (`src/cloaq/`) and may contain known bugs; see [`research/legacy-notebooks/README.md`](../README.md).
> Saved outputs were cleared and local paths removed. QASMBench paths are relative to your own checkout.

In [ ]:
from qiskit import QuantumCircuit
from qiskit import transpile
from qiskit.visualization import plot_histogram
from qiskit_ibm_runtime import SamplerV2
from qiskit_ibm_runtime.fake_provider import FakeManilaV2
from qiskit_aer import AerSimulator
 
# Get a fake backend from the fake provider
# backend = FakeManilaV2()
backend = AerSimulator()

In [ ]:
#from qiskit_ibm_runtime import QiskitRuntimeService
#from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

#backend = QiskitRuntimeService().least_busy(operational=True, simulator=False)
#pass_Manager = generate_preset_pass_manager(optimization_level=1, backend=backend)


In [ ]:
from qiskit import QuantumCircuit
from qiskit.circuit import QuantumRegister, ClassicalRegister
from qiskit.circuit.library import CXGate

def annotate_qk_circuit(circuit, key, real_gate_indices):
    """
    Annotates the locked circuit to visually highlight:
    - qk wire with a label
    - Real key bit CX gates (key=1) marked with labeled barriers
    - Dummy key bit CX gates (key=0) marked differently
    """
    qk = circuit.qregs[-1][0]
    annotated = QuantumCircuit(*circuit.qregs, *circuit.cregs)
    qubit_map = {q: annotated.qubits[i] for i, q in enumerate(circuit.qubits)}
    clbit_map = {old: annotated.clbits[i] for i, old in enumerate(circuit.clbits)}
    key_ptr = 0

    for inst in circuit.data:
        op = inst.operation
        qargs = inst.qubits
        cargs = inst.clbits

        if op.name == "cx" and qargs[0] == qk:
            if key_ptr < len(key):
                key_bit = key[key_ptr]
                label = f"K{key_ptr}={'REAL' if key_bit == 1 else 'DUMMY'}"
                annotated.barrier(*[qubit_map[q] for q in [qargs[0], qargs[1]]])
                labeled_cx = CXGate(label=label)
                annotated.append(labeled_cx,
                                [qubit_map[qargs[0]], qubit_map[qargs[1]]],
                                [])
                annotated.barrier(*[qubit_map[q] for q in [qargs[0], qargs[1]]])
                key_ptr += 1
                continue

        new_qargs = [qubit_map[q] for q in qargs]
        new_cargs = [clbit_map[c] for c in cargs]
        annotated.append(op, new_qargs, new_cargs)

    return annotated


def draw_annotated_circuit(circuit, key, real_gate_indices):
    """
    Creates and draws an annotated version of the locked circuit
    showing key bit positions clearly.
    """
    print("=" * 50)
    print(f"Key bits      : {key}")
    print(f"Real gates    : {real_gate_indices}")
    print(f"Key summary   : {sum(key)} real, {len(key) - sum(key)} dummy")
    print("=" * 50)
    print("\nQLL Gate Legend:")
    for i, bit in enumerate(key):
        status = "REAL  (key=1) — locks circuit" if bit == 1 else "DUMMY (key=0) — identity"
        print(f"  CX #{i}: {status}")

    annotated = annotate_qk_circuit(circuit, key, real_gate_indices)
    return annotated.draw('mpl', style={
        'displaycolor': {
            'cx': '#FF6B6B',
        }
    })


def print_qk_status(stage_name, circuit, key=None, real_gate_indices=None):
    """
    Prints a summary of qk status at any pipeline stage.
    """
    print(f"\n{'='*50}")
    print(f"STAGE: {stage_name}")
    print(f"{'='*50}")
    qk_reg = next((r for r in circuit.qregs if r.name == 'qk'), None)
    print(f"qk present    : {qk_reg is not None}")
    print(f"Num qubits    : {circuit.num_qubits}")
    print(f"Registers     : {circuit.qregs}")
    print(f"Gate counts   : {circuit.count_ops()}")
    if qk_reg is not None:
        qk = qk_reg[0]
        qk_gates = [inst for inst in circuit.data if qk in inst.qubits]
        print(f"Gates on qk   : {len(qk_gates)}")
        for inst in qk_gates:
            print(f"  {inst.operation.name} on "
                  f"{[circuit.find_bit(q).index for q in inst.qubits]}")
    if key is not None:
        print(f"Key           : {key}")
        print(f"Real indices  : {real_gate_indices}")
    qk_meas = [inst for inst in circuit.data
                if inst.operation.name == 'measure'
                and qk_reg is not None
                and qk_reg[0] in inst.qubits]
    print(f"qk measured   : {len(qk_meas) > 0} ({len(qk_meas)} measurements)")

In [ ]:
from qiskit.converters import circuit_to_dag, dag_to_circuit
from qiskit.circuit import QuantumCircuit, QuantumRegister, ClassicalRegister

def track_qk(original_circuit, transpiled_circuit):
    """
    Track where qk lands after transpilation using the layout object.
    Returns the physical qubit index of qk in the transpiled circuit.
    """
    layout = transpiled_circuit.layout
    if layout is None:
        # No remapping occurred, qk is still the last qubit
        return len(transpiled_circuit.qubits) - 1

    qk_qubit = original_circuit.qregs[-1][0]
    try:
        physical_index = layout.initial_layout[qk_qubit]
        print(f"qk (logical) → physical qubit {physical_index}")
        return physical_index
    except Exception:
        # Fallback to last qubit if layout lookup fails
        return len(transpiled_circuit.qubits) - 1


def remove_qk_measurement_by_index(circuit, qk_index):
    """Remove measurement on qk using its tracked physical index."""
    circuit.data = [
        inst for inst in circuit.data
        if not (inst.operation.name == 'measure'
                and circuit.find_bit(inst.qubits[0]).index == qk_index)
    ]
    return circuit


def remove_idle_qwires(circuit, qk_physical_index=None):
    """
    Remove idle qwires, preserve register names, and skip qk measurement.
    If qk_physical_index is provided, uses that to identify qk.
    """
    dag = circuit_to_dag(circuit)
    idle_qubits = set(dag.idle_wires())
    active_qubits = [q for q in dag.qubits if q not in idle_qubits]

    # Identify qk by tracked index or fallback to last register
    if qk_physical_index is not None:
        qk_qubit = circuit.qubits[qk_physical_index]
    else:
        qk_qubit = circuit.qregs[-1][0]
    qk_index = circuit.qubits.index(qk_qubit)

    new_qregs = []
    qubit_map = {}
    for reg in circuit.qregs:
        active_in_reg = [q for q in reg if q in active_qubits]
        if active_in_reg:
            new_reg = QuantumRegister(len(active_in_reg), reg.name)
            new_qregs.append(new_reg)
            for i, q in enumerate(active_in_reg):
                qubit_map[q] = new_reg[i]

    new_cregs = [ClassicalRegister(reg.size, reg.name) for reg in circuit.cregs]
    clbit_map = {old: new_cregs[i][j]
                 for i, reg in enumerate(circuit.cregs)
                 for j, old in enumerate(reg)}

    new_circuit = QuantumCircuit(*new_qregs, *new_cregs)

    for inst in circuit.data:
        if inst.operation.name == 'measure':
            qubit_index = circuit.find_bit(inst.qubits[0]).index
            if qubit_index == qk_index:
                continue

        if all(q in qubit_map for q in inst.qubits):
            new_qargs = [qubit_map[q] for q in inst.qubits]
            new_cargs = [clbit_map[c] for c in inst.clbits]
            new_circuit.append(inst.operation, new_qargs, new_cargs)

    return new_circuit

def remove_qk_register(circuit):
    """
    Returns a copy of the circuit without the qk register,
    dropping any gates that involve qk.
    """
    qk_qubit = circuit.qregs[-1][0]
    
    # Build new circuit with all registers except qk
    new_qregs = [reg for reg in circuit.qregs if reg.name != 'qk']
    new_cregs = [ClassicalRegister(reg.size, reg.name) for reg in circuit.cregs]
    new_circuit = QuantumCircuit(*new_qregs, *new_cregs)

    qubit_map = {q: new_circuit.qubits[i] 
                 for i, q in enumerate(circuit.qubits) 
                 if q != qk_qubit}
    clbit_map = {old: new_circuit.clbits[i] 
                 for i, old in enumerate(circuit.clbits)}

    for inst in circuit.data:
        # Skip any instruction that involves qk
        if qk_qubit in inst.qubits:
            continue
        if all(q in qubit_map for q in inst.qubits):
            new_qargs = [qubit_map[q] for q in inst.qubits]
            new_cargs = [clbit_map[c] for c in inst.clbits]
            new_circuit.append(inst.operation, new_qargs, new_cargs)

    return new_circuit

In [ ]:
#import numpy as np
#import random
#from qiskit import QuantumCircuit

#def prepend_U3_gates(circuit):
    #"""
    #Prepends randomly generated U3 gates to each qubit of the input quantum circuit.

    #Parameters:
    #-----------
    #circuit : qiskit.QuantumCircuit
        #The original quantum circuit to which U3 gates will be prepended.

    #Returns:
    #--------
    #full_circuit : qiskit.QuantumCircuit
        #A new circuit consisting of U3 gates applied to each qubit,
        #followed by the original circuit operations.
    #"""
    # Get the total number of qubits in the input circuit
    #total_qubits = circuit.num_qubits

    # Define a list of possible angles: 0, π/4, π/2, ..., 2π
    #angles = [i * (np.pi / 4) for i in range(9)]

    # Create a new empty quantum circuit with the same number of qubits
    #U3_circuit = QuantumCircuit(total_qubits)

    # Apply a random U3 gate to each qubit
    #for qubit in range(total_qubits):
        #theta = random.choice(angles)
        #phi = random.choice(angles)
        #lam = random.choice(angles)
        #U3_circuit.u(theta, phi, lam, qubit)  # Apply U3 gate with chosen angles

    # Compose the U3 gate circuit in front of the original circuit
    #full_circuit = U3_circuit.compose(circuit)

    #return full_circuit


In [ ]:
import matplotlib.pyplot as plt
import qiskit
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
import random
import numpy as np

# Define a classical bitstring key (not yet used in this snippet)
#key = "011"

# Load a quantum circuit from a QASM benchmark file
qc = QuantumCircuit.from_qasm_file('./QASMBench/small/adder_n4/adder_n4.qasm')
#qc = QuantumCircuit.from_qasm_file('QASMBench/small/basis_trotter_n4/basis_trotter_n4.qasm')
#qc = QuantumCircuit.from_qasm_file('QASMBench/small/basis_change_n3/basis_change_n3.qasm')
#qc = QuantumCircuit.from_qasm_file('QASMBench/small/fredkin_n3/fredkin_n3.qasm')
#qc = QuantumCircuit.from_qasm_file('QASMBench/small/wstate_n3/wstate_n3.qasm')

# Prepend randomly parameterized U3 gates to each qubit in the circuit
#qc = prepend_U3_gates(qc)

# Alternative circuits from the QASMBench suite (commented out for easy switching)

#qc = QuantumCircuit.from_qasm_file('QASMBench/small/basis_change_n3/basis_change_n3.qasm')


# Draw the final circuit using matplotlib (this displays the circuit diagram)
qc.draw('mpl')




In [ ]:
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram

#backend = AerSimulator() using fake backend

# Transpile the circuit using the pass manager


# Look into the transpile function and see if there are any options to preserve the original circuit structure, 
# See how to expand 
#transpiled_circuit = transpile(qc, backend)
transpiled_circuit = transpile(
    qc,
    backend,
    optimization_level=0,           # 0 = no optimization, preserves structure  
    basis_gates=None,               # don't force decomposition into specific gate set
    coupling_map=None,              # ignore device topology constraints
    layout_method='trivial',        # keep qubits in original order, no remapping
    routing_method='none',          # no qubit routing/swapping
    translation_method='translator' # minimal gate translation
)

qk_physical = track_qk(qc, transpiled_circuit)

# Execute the transpiled circuit on the backend
job_sim = backend.run(transpiled_circuit, shots=1024)

# Grab the results from the job
result_sim = job_sim.result()

# Get the measurement counts
counts_unlocked = result_sim.get_counts(transpiled_circuit)
print(counts_unlocked)
plot_histogram(counts_unlocked)

In [ ]:
# Draw Transpiled Ciruit
transpiled_circuit_simp = remove_idle_qwires(transpiled_circuit)
transpiled_circuit_simp.draw('mpl')  

Locking

In [ ]:
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.circuit.library import Barrier
from qiskit.circuit import CircuitInstruction
from qiskit.circuit.library import CXGate
import random

from qiskit import QuantumCircuit, QuantumRegister
from qiskit.circuit.library import Barrier
import random



def transform_circuit(qc, seed=42, verbose=True):
    """
    Integrated QLL + Phase-Wall Locking
    with H-mask limiting on qk
    """

    if seed is not None:
        random.seed(seed)

    # Phase gates
    phase_gates = {'rz', 'p', 's', 't', 'sdg', 'tdg'}

    # Strip barriers
    orig_instrs = [
        inst for inst in qc.data
        if not isinstance(inst.operation, Barrier)
    ]
    m = len(orig_instrs)

    # -------------------------
    # Key generation
    # -------------------------
    n = min(6, m)
    key = [random.randint(0, 1) for _ in range(n)]
    n1 = sum(key)

    real_gate_indices = sorted(random.sample(range(m), n1))

    # -------------------------
    # Build locked circuit
    # -------------------------
    key_qreg = QuantumRegister(1, 'qk')
    locked_qc = QuantumCircuit(*qc.qregs, key_qreg, *qc.cregs)
    qk = key_qreg[0]

    key_ptr = 0
    real_ptr = 0
    last_phase = None

    # ---- H-mask state tracking ----
    qk_in_h_basis = False

    has_emitted_qll_cx = False

    for idx, inst in enumerate(orig_instrs):
        is_phase = inst.operation.name.lower() in phase_gates

        # -------- Phase-wall logic --------
        if last_phase is not None and is_phase != last_phase:
            locked_qc.barrier(*locked_qc.qubits)

        # -------- QLL logic --------
        if key_ptr < n:
            key_bit = key[key_ptr]
            target = inst.qubits[0]

            # ----- REAL controlled gate (key bit = 1) -----
            if (
                key_bit == 1
                and real_ptr < n1
                and idx == real_gate_indices[real_ptr]
            ):
                # ensure H basis
                if not qk_in_h_basis:
                    locked_qc.h(qk)
                    locked_qc.barrier(qk)
                    qk_in_h_basis = True

                # emit CX
                locked_qc.cx(qk, target)

                # emit X unless this is the very first QLL CX
                if has_emitted_qll_cx:
                    locked_qc.x(qk)
                    qk_in_h_basis = False
                else:
                    # first CX: skip X, but still exit H basis logically
                    qk_in_h_basis = False
                    has_emitted_qll_cx = True

                real_ptr += 1
                key_ptr += 1

            # ----- DUMMY controlled gate (key bit = 0) -----
            elif key_bit == 0:
                if not qk_in_h_basis:
                    locked_qc.h(qk)
                    locked_qc.barrier(qk)
                    qk_in_h_basis = True

                locked_qc.cx(qk, target)

                if has_emitted_qll_cx:
                    locked_qc.x(qk)
                    qk_in_h_basis = False
                else:
                    qk_in_h_basis = False
                    has_emitted_qll_cx = True

                # Second dummy CNOT (identity overall)
                locked_qc.cx(qk, target)

                key_ptr += 1

        # -------- Original gate --------
        locked_qc.append(inst.operation, inst.qubits, inst.clbits)

        # Barrier after every original Hadamard
        if inst.operation.name.lower() == "h":
            locked_qc.barrier(*inst.qubits)
            
        last_phase = is_phase

    if last_phase:
        locked_qc.barrier(*locked_qc.qubits)

    # -------------------------
    # Debug output
    # -------------------------
    if verbose:
        print("---- Integrated QLL + Phase-Wall Locking ----")
        print(f"Key            : {key}")
        print(f"Real gates     : {real_gate_indices}")
        print(f"Original gates : {m}")
        print(f"Locked gates   : {len(locked_qc.data)}")

    return locked_qc, key, {
        "real_gate_indices": real_gate_indices
    }

In [ ]:
converted_circuit, key, meta = transform_circuit(qc)
converted_circuit.draw('mpl')

In [ ]:
converted_circuit, key, meta = transform_circuit(qc)
print_qk_status("After transform_circuit", converted_circuit, key, meta['real_gate_indices'])
draw_annotated_circuit(converted_circuit, key, meta['real_gate_indices'])

In [ ]:
def remove_qk_measurement(circuit):
    qk_qubit = circuit.qregs[-1][0]
    qk_index = circuit.qubits.index(qk_qubit)
    circuit.data = [
        inst for inst in circuit.data
        if not (inst.operation.name == 'measure'
                and circuit.find_bit(inst.qubits[0]).index == qk_index)
    ]
    return circuit

def restore_qk_name(circuit):
    """
    After transpilation, split the last qubit into its own 'qk' register
    while keeping all other qubits in a single 'q' register.
    """
    all_qubits = circuit.qubits
    main_qubits = all_qubits[:-1]  # all except last
    qk_qubit = all_qubits[-1]      # last qubit

    main_reg = QuantumRegister(len(main_qubits), 'q')
    qk_reg = QuantumRegister(1, 'qk')
    new_cregs = [ClassicalRegister(reg.size, reg.name) for reg in circuit.cregs]

    new_circuit = QuantumCircuit(main_reg, qk_reg, *new_cregs)

    qubit_map = {old: new_circuit.qubits[i] for i, old in enumerate(all_qubits)}
    clbit_map = {old: new_circuit.clbits[i] for i, old in enumerate(circuit.clbits)}

    for inst in circuit.data:
        new_qargs = [qubit_map[q] for q in inst.qubits]
        new_cargs = [clbit_map[c] for c in inst.clbits]
        new_circuit.append(inst.operation, new_qargs, new_cargs)

    return new_circuit

In [ ]:
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram

#backend = AerSimulator()

# Apply it right before transpile
qk_index = converted_circuit.qubits.index(converted_circuit.qregs[-1][0])
converted_circuit = remove_qk_measurement_by_index(converted_circuit, qk_index)

# Transpile the circuit using the pass manager
transpiled_circuit = transpile(converted_circuit, backend)

# Track qk through transpilation
qk_physical = track_qk(converted_circuit, transpiled_circuit)
transpiled_circuit = restore_qk_name(transpiled_circuit)

# Execute the transpiled circuit on the backend
job_sim = backend.run(transpiled_circuit, shots=1024)

# Grab the results from the job
result_sim = job_sim.result()

# Get the measurement counts
counts_original = result_sim.get_counts(transpiled_circuit)
print(counts_original)
plot_histogram(counts_original)

In [ ]:
from qiskit.circuit import QuantumRegister, ClassicalRegister
from qiskit import QuantumCircuit

# After transpile, restore the name
transpiled_circuit = transpile(converted_circuit, backend)
transpiled_circuit = restore_qk_name(transpiled_circuit)

# Then remove idle wires + draw as normal
transpiled_circuit_simp = remove_idle_qwires(transpiled_circuit)
print_qk_status("After remove_idle_qwires", transpiled_circuit_simp)
transpiled_circuit_simp.draw('mpl')


In [ ]:
from math import pi

def key_phase(key):
    """
    Maps a 3-bit binary string to a specific phase angle (in radians) based on π.

    Parameters:
    -----------
    key : str
        A 3-bit binary string (e.g., '000', '011', '111').

    Returns:
    --------
    float
        A phase angle in radians, ranging from 0.25π to 2π, depending on the input key.

    Mapping:
    --------
        '000' → 0.25π
        '001' → 0.5π
        '010' → 0.75π
        '011' → 1.0π
        '100' → 1.25π
        '101' → 1.5π
        '110' → 1.75π
        '111' → 2.0π
    """
    match key:
        case '000':
            return 0.25 * pi
        case '001':
            return 0.5 * pi
        case '010':
            return 0.75 * pi
        case '011':
            return 1.0 * pi
        case '100':
            return 1.25 * pi
        case '101':
            return 1.5 * pi
        case '110':
            return 1.75 * pi
        case '111':
            return 2.0 * pi


In [ ]:
from math import pi
from qiskit import QuantumCircuit
from qiskit.circuit import ClassicalRegister 
from qiskit import QuantumRegister
import random


def replace_with_phase_gates(circuit, key_phase_offset=0):
    """
    Replace Z, S, T, and their inverses with custom phase gates.
    Compatible with QLL-locked circuits (preserves qk register and CX gates).
    """
    new_circuit = QuantumCircuit(*circuit.qregs, *circuit.cregs)

    qubit_map = {old: new for old, new in zip(circuit.qubits, new_circuit.qubits)}
    clbit_map = {old: new for old, new in zip(circuit.clbits, new_circuit.clbits)}

    for instruction in circuit.data:               
        gate   = instruction.operation
        qubits = instruction.qubits
        clbits = instruction.clbits

        q_new = [qubit_map[q] for q in qubits]
        c_new = [clbit_map[c] for c in clbits] if clbits else []

        if gate.name == "z":
            new_circuit.p(pi + key_phase_offset, q_new[0])
        elif gate.name == "s":
            new_circuit.p(pi/2 + key_phase_offset, q_new[0])
        elif gate.name == "t":
            new_circuit.p(pi/4 + key_phase_offset, q_new[0])
        elif gate.name == "sdg":
            new_circuit.p(-pi/2 + key_phase_offset, q_new[0])
        elif gate.name == "tdg":
            new_circuit.p(-pi/4 + key_phase_offset, q_new[0])
        else:
            new_circuit.append(gate, q_new, c_new)

    return new_circuit

In [ ]:
phase_converted_circuit = replace_with_phase_gates(converted_circuit)
# phase_converted_circuit.draw('mpl')
#obfuscated_circuit = prepend_U3_gates(phase_converted_circuit)
#obfuscated_circuit.draw('mpl')
phase_converted_circuit.draw('mpl')

In [ ]:
phase_converted_circuit = replace_with_phase_gates(converted_circuit)
print_qk_status("After replace_with_phase_gates", phase_converted_circuit)
phase_converted_circuit.draw('mpl')

In [ ]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister

def add_measurements_from_info(circuit, measurement_info, exclude_qk=True):
    """
    Adds measurement gates to a quantum circuit based on the provided measurement information.

    Parameters:
    circuit (QuantumCircuit): The quantum circuit to which measurements will be added.
    measurement_info (dict): A dictionary containing measurement information, as returned by `get_measurement_info`.
    exclude_qk (bool): If True, don't measure the last qubit (assumed to be qk).

    Returns:
    QuantumCircuit: The modified quantum circuit with measurement gates added.
    """
    # Identify qk if excluding
    qk_index = None
    if exclude_qk and len(circuit.qregs) > 0:
        # Assume qk is the last register 
        qk_index = circuit.qubits.index(circuit.qregs[-1][0])
    
    # Iterate over the measurement information
    for qubit, info in measurement_info.items():
        qubit_index = info['measured_qubit']
        clbit_index = info['classical_bit']
        
        # Skip qk if excluding
        if exclude_qk and qubit_index == qk_index:
            continue
        
        # Add the measurement gate to the circuit
        circuit.measure(qubit_index, clbit_index)
    
    return circuit


def remove_all_measurements(circuit):
    """
    Removes all measurement operations from a quantum circuit,
    preserving all quantum and classical registers exactly.
    """
    new_qregs = [QuantumRegister(reg.size, reg.name) for reg in circuit.qregs]
    new_cregs = [ClassicalRegister(reg.size, reg.name) for reg in circuit.cregs]
    new_circuit = QuantumCircuit(*new_qregs, *new_cregs)

    qubit_map = {}
    for old_reg, new_reg in zip(circuit.qregs, new_qregs):
        for i in range(old_reg.size):
            qubit_map[old_reg[i]] = new_reg[i]

    clbit_map = {}
    for old_reg, new_reg in zip(circuit.cregs, new_cregs):
        for i in range(old_reg.size):
            clbit_map[old_reg[i]] = new_reg[i]

    for instruction in circuit.data:
        inst = instruction.operation
        qargs = instruction.qubits
        cargs = instruction.clbits
        
        if inst.name == 'measure':
            continue

        new_q = [qubit_map[q] for q in qargs]
        new_c = [clbit_map[c] for c in cargs]

        new_circuit.append(inst, new_q, new_c)

    return new_circuit


def get_measurement_info(circuit, exclude_qk=True):
    """
    Analyzes a quantum circuit and provides information about measurement gates.
    Optionally excludes the key qubit (qk) from the returned info.

    Parameters:
    circuit (QuantumCircuit): The quantum circuit to analyze.
    exclude_qk (bool): If True, don't include qk measurements in the returned info.

    Returns:
    dict: A dictionary containing information about the measurement gates.
    """
    measurement_info = {}
    
    # Identify qk if excluding
    qk_index = None
    if exclude_qk and len(circuit.qregs) > 0:
        qk_index = circuit.qubits.index(circuit.qregs[-1][0])
    
    for instruction in circuit.data:
        inst = instruction.operation
        qubits = instruction.qubits
        clbits = instruction.clbits
        
        if inst.name == 'measure':
            qubit_index = circuit.find_bit(qubits[0]).index
            clbit_index = circuit.find_bit(clbits[0]).index
            
            # Skip qk if excluding
            if exclude_qk and qubit_index == qk_index:
                continue
            
            measurement_info[qubit_index] = {
                'measured_qubit': qubit_index,
                'measurement_gate': 'M',
                'classical_bit': clbit_index
            }
    
    return measurement_info


measurement_info = get_measurement_info(phase_converted_circuit, exclude_qk=True)

print("Measurement Information (excluding qk):")
for qubit, info in measurement_info.items():
    print(f"Qubit {info['measured_qubit']} is measured using gate '{info['measurement_gate']}' "
          f"and the result is stored in classical bit {info['classical_bit']}.")

In [ ]:
def add_random_phase_layers_with_reassembly(circuit, num_layers, num_splits):
    """
    Inserts random phase layers at split points.
    Compatible with QLL-locked circuits (preserves qk register).
    """
    phases = [0.25*pi, 0.5*pi, 0.75*pi, 1*pi, 1.25*pi, 1.5*pi, 1.75*pi, 2*pi]

    new_qregs = [QuantumRegister(qreg.size, qreg.name) for qreg in circuit.qregs]
    new_cregs = [ClassicalRegister(creg.size, creg.name) for creg in circuit.cregs]

    new_circuit = QuantumCircuit(*new_qregs, *new_cregs)

    qubit_map = {}
    for old_reg, new_reg in zip(circuit.qregs, new_qregs):
        for i in range(old_reg.size):
            qubit_map[old_reg[i]] = new_reg[i]

    clbit_map = {}
    for old_reg, new_reg in zip(circuit.cregs, new_cregs):
        for i in range(old_reg.size):
            clbit_map[old_reg[i]] = new_reg[i]

    ops = [
        (inst.operation, inst.qubits, inst.clbits)
        for inst in circuit.data
        if inst.operation.name != "measure"
    ]

    split_points = [len(ops) // (num_splits + 1) * (i + 1) for i in range(num_splits)]

    last_gate_name = None

    def safe_append(inst, q, c):
        nonlocal last_gate_name

        if inst.name == "barrier":
            if last_gate_name != "barrier":
                new_circuit.barrier(*new_circuit.qubits)
                last_gate_name = "barrier"
            return

        if inst.name == last_gate_name and inst.name in ("h", "cx"):
            return

        q_new = [qubit_map[x] for x in q]
        c_new = [clbit_map[x] for x in c]

        new_circuit.append(inst, q_new, c_new)
        last_gate_name = inst.name

    for i, (inst, q, c) in enumerate(ops):
        safe_append(inst, q, c)

        if i + 1 in split_points:
            for _ in range(num_layers):
                new_circuit.barrier(*new_circuit.qubits)
                last_gate_name = "barrier"
                for qb in new_circuit.qubits:
                    new_circuit.p(random.choice(phases), qb)
                new_circuit.barrier(*new_circuit.qubits)
                last_gate_name = "barrier"

    # Identify qk (last register) and measure all qubits except it
    qk_qubit = new_circuit.qregs[-1][0]
    for i, qb in enumerate(new_circuit.qubits):
        if qb != qk_qubit:
            new_circuit.measure(qb, i)

    return new_circuit


random_layer_added_qc = add_random_phase_layers_with_reassembly(remove_all_measurements(phase_converted_circuit), 1, 2)
random_layer_added_qc.draw('mpl')

In [ ]:
random_layer_added_qc = add_random_phase_layers_with_reassembly(remove_all_measurements(phase_converted_circuit), 1,2)
random_layer_added_qc.draw('mpl')

In [ ]:
random_layer_added_qc = add_random_phase_layers_with_reassembly(
    remove_all_measurements(phase_converted_circuit), 1, 2)
print_qk_status("After random_phase_layers", random_layer_added_qc)
random_layer_added_qc.draw('mpl')

In [ ]:
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram

# Use AerSimulator as the backend
#backend = AerSimulator()

# Transpile the circuit using the pass manager
transpiled_circuit = transpile(random_layer_added_qc, backend)
transpiled_circuit = restore_qk_name(transpiled_circuit)

# Execute the transpiled circuit on the backend
job_sim = backend.run(transpiled_circuit, shots=1024)

# Grab the results from the job
result_sim = job_sim.result()

# Get the measurement counts
counts_locked = result_sim.get_counts(transpiled_circuit)
print(counts_locked)
plot_histogram(counts_locked)

In [ ]:
transpiled_circuit_simp = remove_idle_qwires(transpiled_circuit, qk_physical_index=qk_physical)
transpiled_circuit_simp.draw('mpl')  

TVD Locked

In [ ]:
# import numpy as np
# def total_variation_distance_from_counts(counts1, counts2, shots = 1024):
#     """
#     Calculate the Total Variation Distance (TVD) between two probability distributions
#     derived from Qiskit counts.

#     Parameters:
#     counts1 (dict): The counts from the first circuit.
#     counts2 (dict): The counts from the second circuit.

#     Returns:
#     float: The Total Variation Distance between the two distributions.
#     """
#     # Get all possible keys (states) from both counts
#     all_states = set(counts1.keys()).union(set(counts2.keys()))
    
#     # Normalize counts to get probability distributions
#     total_counts1 = sum(counts1.values())
#     total_counts2 = sum(counts2.values())
    
#     # Initialize probability distributions
#     p = np.zeros(len(all_states))
#     q = np.zeros(len(all_states))
   
#     # Map states to indices
#     state_to_index = {state: i for i, state in enumerate(all_states)}
    
#     # Fill probability distributions
#     for state, count in counts1.items():
#         p[state_to_index[state]] = count 
#     for state, count in counts2.items():
#         q[state_to_index[state]] = count 
  
#     # Calculate the Total Variation Distance
#     tvd = (1/(2*1024)) * np.sum(np.abs(p - q))
#     return tvd

# tvd = total_variation_distance_from_counts(counts_original, counts_locked)
# print("Counts Original:", counts_original)
# print("Counts Locked:", counts_locked)
# print(f"Total Variation Distance: {tvd:.6f}")
# print(f"key: {key}")

Unlocking

In [ ]:
# from qiskit import QuantumCircuit
# from qiskit.circuit.library import RZGate, PhaseGate
# from qiskit.circuit import Barrier

# def subtract_phase_offset(circuit: QuantumCircuit, offset: float) -> QuantumCircuit:
#     """
#     Subtracts a phase offset from all RZGate and PhaseGate instructions within barriers.

#     Args:
#         circuit (QuantumCircuit): The input quantum circuit.
#         offset (float): The phase offset to subtract.

#     Returns:
#         QuantumCircuit: The modified quantum circuit.
#     """
#     # Create a new circuit with the same number of qubits and classical bits
#     new_circuit = QuantumCircuit(circuit.num_qubits, circuit.num_clbits)

#     # Copy over all the registers from the original circuit
#     for qreg in circuit.qregs:
#         new_circuit.add_register(qreg)
#     for creg in circuit.cregs:
#         new_circuit.add_register(creg)

#     # Flag to indicate if we are inside a barrier
#     inside_barrier = False

#     # Iterate over the instructions in the circuit
#     for instruction, qargs, cargs in circuit.data:
#         # Check if the instruction is a barrier
#         if isinstance(instruction, Barrier):
#             inside_barrier = not inside_barrier  # Toggle the flag
#             new_circuit.append(instruction, qargs, cargs)  # Add the barrier to the new circuit
#         elif inside_barrier and (isinstance(instruction, RZGate) or isinstance(instruction, PhaseGate)):
#             # If inside a barrier and the instruction is a phase gate, modify its phase
#             original_phase = instruction.params[0]
#             new_phase = original_phase - offset
#             # Add the modified phase gate to the new circuit
#             if isinstance(instruction, RZGate):
#                 new_circuit.rz(new_phase, *qargs)
#             elif isinstance(instruction, PhaseGate):
#                 new_circuit.p(new_phase, *qargs)
#         else:
#             # Otherwise, add the instruction as is
#             new_circuit.append(instruction, qargs, cargs)

#     return new_circuit  # Return the modified circuit

# from qiskit import QuantumCircuit
# from qiskit.transpiler import PassManager
# from qiskit.transpiler.passes import RemoveBarriers, DAGFixedPoint
# from qiskit.converters import circuit_to_dag, dag_to_circuit

# def remove_gates_inside_barriers(circuit):
#     # Convert the circuit to a DAG (Directed Acyclic Graph)
#     dag = circuit_to_dag(circuit)
    
#     # Iterate over all nodes in the DAG
#     for node in dag.op_nodes():
#         # Check if the node is a barrier
#         if node.name == 'barrier':
#             # Get the qubits involved in the barrier
#             barrier_qubits = node.qargs
            
#             # Remove all gates that are inside the barrier
#             for successor in list(dag.successors(node)):
#                 if all(qubit in barrier_qubits for qubit in successor.qargs):
#                     print(f"Removing gate {successor.name} inside barrier")
#                     # dag.remove_op_node(successor)
    
#     # Convert the modified DAG back to a QuantumCircuit
#     new_circuit = dag_to_circuit(dag)
    
#     return new_circuit

# def modify_rz_inside_barriers(circuit, phase_shift):
#     """
#     Modify the phase of Rz gates inside barriers by adding a phase shift.
    
#     Args:
#         circuit (QuantumCircuit): The input quantum circuit.
#         phase_shift (float): The phase shift to apply to Rz gates inside barriers.
    
#     Returns:
#         QuantumCircuit: The modified quantum circuit.
#     """
#     # Convert the circuit to a DAG (Directed Acyclic Graph)
#     dag = circuit_to_dag(circuit)
    
#     # Track whether we are inside a barrier
#     inside_barrier = False
#     dummy_gate_count = 0
    
#     # Iterate over all nodes in the DAG
#     for node in dag.topological_op_nodes():
#         # Check if the node is a barrier
#         if node.name == 'barrier':
#             inside_barrier = not inside_barrier  # Toggle the inside_barrier flag
#             continue
        
#         # If inside a barrier and the node is an Rz gate, modify its phase
#         if inside_barrier and node.name == 'rz':
#             # Get the current phase of the Rz gate
#             current_phase = node.op.params[0]
#             # Apply the phase shift
#             new_phase = current_phase + phase_shift
#             # Update the Rz gate with the new phase
#             node.op.params[0] = 0

#             dummy_gate_count += 1
#             print (f" Dummy gate created: {node.name} on qubit {node.qargs[0]} (original phase = {current_phase})")
#     # Convert the modified DAG back to a QuantumCircuit
#     modified_circuit = dag_to_circuit(dag)
#     print(f"\nTotal dummy gates created: {dummy_gate_count}\n")
#     return modified_circuit

In [ ]:
from qiskit import QuantumCircuit
from qiskit.transpiler import PassManager
from qiskit.transpiler.passes import RemoveBarriers, DAGFixedPoint
from qiskit.converters import circuit_to_dag, dag_to_circuit
from qiskit.circuit import Barrier  


# ------------------------------------------------------------
# PHASE-WALL LOCKING 
# ------------------------------------------------------------
def modify_rz_inside_barriers(circuit: QuantumCircuit, phase_shift: float):
    """
    Adds a phase shift to all RZ gates inside barrier regions.
    This is the ACTIVE LOCK used before decryption.
    """

    new_circuit = QuantumCircuit(*circuit.qregs, *circuit.cregs)

    qubit_map = {q: new_circuit.qubits[i] for i, q in enumerate(circuit.qubits)}
    clbit_map = {old: new_circuit.clbits[i] for i, old in enumerate(circuit.clbits)}

    inside_barrier = False

    for inst in circuit.data:
        op = inst.operation
        qargs = inst.qubits
        cargs = inst.clbits

        if op.name == 'barrier':
            inside_barrier = not inside_barrier
            new_circuit.barrier(*[qubit_map[q] for q in qargs])
            continue

        if inside_barrier and op.name == 'rz':
            new_circuit.rz(op.params[0] + phase_shift, qubit_map[qargs[0]])
        else:
            new_circuit.append(op, [qubit_map[q] for q in qargs],
                               [clbit_map[c] for c in cargs])

    return new_circuit


# ------------------------------------------------------------
# PHASE-WALL DECRYPTION
# ------------------------------------------------------------
def subtract_phase_offset(circuit: QuantumCircuit, offset: float) -> QuantumCircuit:
    """
    Inverse of modify_rz_inside_barriers.
    Subtracts the phase offset from Phase / RZ gates inside barriers.
    """

    new_circuit = QuantumCircuit(*circuit.qregs, *circuit.cregs)
    inside_barrier = False

    for inst in circuit.data:
        instruction = inst.operation
        qargs = inst.qubits
        cargs = inst.clbits

        # Handle barriers
        if instruction.name == 'barrier':
            inside_barrier = not inside_barrier
            new_circuit.append(instruction, qargs, cargs)
            continue

        # Handle phase gates INSIDE barriers
        if inside_barrier and instruction.name in ['rz', 'p']:
            theta = instruction.params[0]

            if instruction.name == 'rz':
                new_circuit.rz(theta - offset, qargs[0])
            else:  # 'p'
                new_circuit.p(theta - offset, qargs[0])

            continue

        # Default: copy gate
        new_circuit.append(instruction, qargs, cargs)

    return new_circuit


# ------------------------------------------------------------
# QLL DECRYPTION 
# ------------------------------------------------------------
def decrypt_qll(locked_circuit: QuantumCircuit, qll_key) -> QuantumCircuit:
    """
    QLL Decryption following the paper's Dec(Circ', k) process:

    - Removes H gates on qk
    - Inserts Pauli-X gates on qk wherever key bits differ
      (this prepares qk in the correct classical state before each CX)
    - Keeps all CX gates in place (controlled by qk)
    - Does NOT remove qk — the circuit retains the key qubit and CX structure
    
    After this step the CX gates are deterministic:
      qk == |1> => CX acts as X on target  
      qk == |0> => CX acts as identity  
    """

    qk = locked_circuit.qregs[-1][0]

    # Output circuit keeps ALL registers including the key qubit
    qc = QuantumCircuit(*locked_circuit.qregs, *locked_circuit.cregs)

    qubit_map = {q: qc.qubits[i] for i, q in enumerate(locked_circuit.qubits)}
    clbit_map = {old: qc.clbits[i] for i, old in enumerate(locked_circuit.clbits)}

    key_ptr = 0
    # Track whether we just handled a dummy double-CX
    last_cx_was_dummy = False

    for instruction in locked_circuit.data:
        op = instruction.operation
        qargs = instruction.qubits
        cargs = instruction.clbits

        # Drop barriers
        if op.name == "barrier":
            continue

        # Remove H gates on qk
        if op.name == "h" and qk in qargs:
            continue

        if op.name == "x" and qk in qargs:
            continue

        if op.name == "cx" and qargs[0] == qk:
            if last_cx_was_dummy:
                # This is the second CX of a dummy pair — skip it, don't advance key_ptr
                last_cx_was_dummy = False
                continue

            if key_ptr < len(qll_key):
                key_bit = qll_key[key_ptr]
                if key_bit == 1:
                    qc.x(qubit_map[qargs[1]])
                    last_cx_was_dummy = False
                else:
                    # Dummy: next CX belongs to this pair
                    last_cx_was_dummy = True
                key_ptr += 1
            continue
        
        if qk in qargs:
            continue

        new_qargs = [qubit_map[q] for q in qargs]
        new_cargs = [clbit_map[c] for c in cargs]
        qc.append(op, new_qargs, new_cargs)

    return qc

# ------------------------------------------------------------
# FULL DECRYPTION PIPELINE (QLL + PHASE-WALL)
# ------------------------------------------------------------
def decrypt_full_circuit(
    locked_circuit: QuantumCircuit,
    qll_key,
    phase_offset: float
):
    """
    Fully decrypts a circuit locked with:
      - QLL control-path locking
      - Barrier-based phase-wall locking
    """

    # Step 1: QLL unlock
    qc = decrypt_qll(locked_circuit, qll_key)

    # Step 2: Phase-wall unlock
    qc = subtract_phase_offset(qc, phase_offset)

    # Step 3: Barrier cleanup
    pm = PassManager(RemoveBarriers())
    qc = pm.run(qc)
    
    # Step 4: Strip qk register from final output
    qk = qc.qregs[-1][0]
    new_qregs = [reg for reg in qc.qregs if reg.name != 'qk']
    new_circuit = QuantumCircuit(*new_qregs, *qc.cregs)

    qubit_map = {q: new_circuit.qubits[i]
                 for i, q in enumerate(qc.qubits)
                 if q != qk}
    clbit_map = {old: new_circuit.clbits[i]
                 for i, old in enumerate(qc.clbits)}

    for inst in qc.data:
        if qk in inst.qubits:
            continue
        new_qargs = [qubit_map[q] for q in inst.qubits]
        new_cargs = [clbit_map[c] for c in inst.clbits]
        new_circuit.append(inst.operation, new_qargs, new_cargs)

    return new_circuit

In [ ]:
circuit_for_locking = remove_qk_register(transpiled_circuit_simp)

modified_circuit, qll_key, lock_info = transform_circuit(circuit_for_locking, seed=42)
print(f"qregs after transform: {modified_circuit.qregs}")

p_offset = key_phase("010")
modified_circuit = modify_rz_inside_barriers(modified_circuit, -p_offset)
print(f"qregs after modify_rz: {modified_circuit.qregs}")

decrypted_circuit = decrypt_full_circuit(
    locked_circuit=modified_circuit,
    qll_key=qll_key,
    phase_offset=p_offset
)

decrypted_circuit.draw('mpl')

For the decryption the CNOT gate has to be controlled by an existing gates

So the CNOT will change to a CCNOT gate to make it real

goes for any key = 1

In [ ]:
from qiskit import transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
import numpy as np
import matplotlib.pyplot as plt
import random
import time

# --- TVD Function Definition ---
def total_variation_distance_from_counts(counts1, counts2, shots=1024, plot=True, labels=None):
    """
    Calculate TVD per the paper's formula:

        TVD = sum_i( |yi,lock - yi,orig| ) / (2 * N * 2^b)

    Where:
        N   = number of shots
        b   = number of bits in the output (inferred from state length)
        yi  = raw count of outcome i in each distribution
    
    Parameters:
    -----------
    counts1 : dict
        First distribution (e.g., original circuit counts)
    counts2 : dict
        Second distribution (e.g., locked circuit counts)
    shots : int
        Number of shots used in the simulation
    plot : bool
        If True, display histograms of both distributions
    labels : list of str
        Labels for the two distributions (default: ["Distribution 1", "Distribution 2"])
    
    Returns:
    --------
    float : TVD value
    """
    
    if labels is None:
        labels = ["Distribution 1", "Distribution 2"]

    # All possible output states across both distributions
    all_states = set(counts1.keys()).union(set(counts2.keys()))

    # Infer number of bits from state string length
    b = len(next(iter(all_states)))

    # Map states to indices
    state_to_index = {state: i for i, state in enumerate(all_states)}

    # Fill raw count vectors (missing states = 0 counts)
    p = np.zeros(len(all_states))
    q = np.zeros(len(all_states))

    for state, count in counts1.items():
        p[state_to_index[state]] = count
    for state, count in counts2.items():
        q[state_to_index[state]] = count

    # TVD = sum|yi,lock - yi,orig| / (2 * N * 2^b)
    tvd = np.sum(np.abs(p - q)) / (2 * shots * (2 ** b))

    if plot:
        fig = plot_histogram([counts1, counts2], legend=labels, figsize=(12, 6))
        display(fig)  
         

    return tvd 

# --- Simulation and TVD Calculation ---
# --- Simulation and TVD Calculation ---
backend = AerSimulator()

# Step 1: Simulate ORIGINAL circuit
transpiled_original = transpile(circuit_for_locking, backend)
qk_physical_orig = track_qk(circuit_for_locking, transpiled_original)
transpiled_original = remove_qk_measurement_by_index(transpiled_original, qk_physical_orig)
job_original = backend.run(transpiled_original, shots=1024)
counts_original = job_original.result().get_counts(transpiled_original)

# Step 2: Fix measurements on already-decrypted circuit (reuse from previous cell)
decrypted_clean = remove_all_measurements(decrypted_circuit)
measurement_info = get_measurement_info(circuit_for_locking, exclude_qk=False)
decrypted_clean = add_measurements_from_info(decrypted_clean, measurement_info, exclude_qk=True)

# Step 3: Simulate DECRYPTED circuit
transpiled_decrypted = transpile(decrypted_clean, backend)
qk_physical_dec = track_qk(decrypted_clean, transpiled_decrypted)
transpiled_decrypted = remove_qk_measurement_by_index(transpiled_decrypted, qk_physical_dec)
job_decrypted = backend.run(transpiled_decrypted, shots=1024)
counts_decrypted = job_decrypted.result().get_counts(transpiled_decrypted)

# Step 4: Calculate TVD
tvd = total_variation_distance_from_counts(
    counts_original,
    counts_decrypted,
    shots=1024,
    plot=True,
    labels=["Original Circuit", "Decrypted Circuit"]
)

print("Counts Original: ", counts_original)
print("Counts Decrypted:", counts_decrypted)
print(f"Total Variation Distance: {tvd:.6f}")
print(f"Key: {qll_key}")

Check Circuits of original and decrypted to check logic
Should have the same distributions

In [ ]:
print("=== Decrypted circuit instruction order ===")
for i, inst in enumerate(decrypted_circuit.data):
    print(f"{i:3}: {inst.operation.name:8} on {[decrypted_circuit.find_bit(q).index for q in inst.qubits]}")

print("\n=== Original circuit instruction order ===")
for i, inst in enumerate(circuit_for_locking.data):
    if inst.operation.name != 'measure':
        print(f"{i:3}: {inst.operation.name:8} on {[circuit_for_locking.find_bit(q).index for q in inst.qubits]}")

In [ ]:
qk = modified_circuit.qregs[-1][0]
key_ptr = 0
current_key = 0
cx_original = 0
cx_qll = 0
x_added = 0

for instruction in modified_circuit.data:
    op = instruction.operation
    qargs = instruction.qubits

    if op.name == "barrier":
        continue
    if op.name == "h" and qk in qargs:
        continue

    if op.name == "cx" and qargs[0] == qk:
        if key_ptr < len(qll_key):
            next_key = qll_key[key_ptr]
            print(f"QLL CX #{key_ptr}: key_bit={next_key}, target={qargs[1]} "
                  f"→ {'X on target' if next_key == 1 else 'dropped'}")
            if next_key == 1:
                x_added += 1
            cx_qll += 1
            current_key = next_key
            key_ptr += 1
        continue  # always drop QLL CX

    if qk in qargs:
        continue

    if op.name == "cx":
        cx_original += 1

print(f"\nQLL CX gates dropped : {cx_qll}")
print(f"X gates added        : {x_added}")
print(f"Original CX kept     : {cx_original}")
print(f"\nExpected original cx : {circuit_for_locking.count_ops().get('cx', 0)}")
print(f"Match: {cx_original == circuit_for_locking.count_ops().get('cx', 0)}")


print("=== Gate count comparison ===")
print(f"Original        : {circuit_for_locking.count_ops()}")
print(f"Decrypted       : {decrypted_circuit.count_ops()}")

print("\n=== Qubit by qubit measurement check ===")
for i, q in enumerate(decrypted_circuit.qubits):
    print(f"qubit {i}: {decrypted_circuit.qregs}")

print("\n=== Counts comparison ===")
print(f"Original  : {counts_original}")
print(f"Decrypted : {counts_decrypted}")

In [ ]:
# Check what circuit_for_locking actually looks like
print("=== circuit_for_locking ===")
print(f"Qregs: {circuit_for_locking.qregs}")
print(f"Cregs: {circuit_for_locking.cregs}")
print(f"Num qubits: {circuit_for_locking.num_qubits}")
print(f"Num clbits: {circuit_for_locking.num_clbits}")
print(f"Gate counts: {circuit_for_locking.count_ops()}")
print(f"Measurements: {get_measurement_info(circuit_for_locking, exclude_qk=False)}")

print("\n=== decrypted_circuit ===")
print(f"Qregs: {decrypted_circuit.qregs}")
print(f"Cregs: {decrypted_circuit.cregs}")
print(f"Num qubits: {decrypted_circuit.num_qubits}")
print(f"Num clbits: {decrypted_circuit.num_clbits}")
print(f"Gate counts: {decrypted_circuit.count_ops()}")

print("\n=== original qc ===")
print(f"Qregs: {qc.qregs}")
print(f"Cregs: {qc.cregs}")
print(f"Gate counts: {qc.count_ops()}")
print(f"Measurements: {get_measurement_info(qc, exclude_qk=False)}")

In [ ]:
from qiskit import transpile 
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram

# Use AerSimulator as the backend
backend = AerSimulator()

# Transpile the circuit using the pass manager
transpiled_circuit = transpile(modified_circuit, backend)

# Execute the transpiled circuit on the backend
job_sim = backend.run(transpiled_circuit, shots=1024)

# Grab the results from the job
result_sim = job_sim.result()

# Get the measurement counts
counts = result_sim.get_counts(transpiled_circuit)
print(counts)
plot_histogram(counts)


In [ ]:
from qiskit import QuantumCircuit
from qiskit.circuit.library import PhaseGate
from qiskit.circuit import Barrier

def subtract_phase_offset(circuit: QuantumCircuit, offset: float) -> QuantumCircuit:
    """
    This function takes a quantum circuit, looks for barriers, and inside each barrier,
    it looks for phase gates and subtracts an offset from their phase.

    Args:
        circuit (QuantumCircuit): The input quantum circuit.
        offset (float): The offset to subtract from the phase of phase gates.

    Returns:
        QuantumCircuit: The modified quantum circuit.
    """
    # Create a new circuit to store the modified instructions
    new_circuit = QuantumCircuit(*circuit.qregs, *circuit.cregs)

    # Flag to indicate if we are inside a barrier
    inside_barrier = False

    # Iterate over the instructions in the circuit
    for instruction, qargs, cargs in circuit.data:
        # Check if the instruction is a barrier
        if isinstance(instruction, Barrier):
            inside_barrier = not inside_barrier  # Toggle the flag
            new_circuit.append(instruction, qargs, cargs)  # Add the barrier to the new circuit
        elif inside_barrier and isinstance(instruction, PhaseGate):
            # If inside a barrier and the instruction is a phase gate, modify its phase
            original_phase = instruction.params[0]
            new_phase = original_phase - offset
            new_circuit.p(new_phase, *qargs)  # Add the modified phase gate
        else:
            # Otherwise, add the instruction as is
            new_circuit.append(instruction, qargs, cargs)

    return new_circuit

# Example usage:
#qc = QuantumCircuit(2)
#qc.p(0.5, 0)  # Add a phase gate with phase 0.5
#qc.barrier()  # Add a barrier
#qc.p(0.8, 1)  # Add another phase gate with phase 0.8
#qc.barrier()  # Add another barrier

#print("Original Circuit:")
#print(qc)

# Subtract an offset of 0.2 from the phase gates within barriers
#modified_qc = subtract_phase_offset(qc, 0.2)

#print("\nModified Circuit:")
#print(modified_qc)

In [ ]:
# import numpy as np
# import matplotlib.pyplot as plt

# # Load original circuit from QASM file 
# qc_original = qc

# # Create locked circuit by transforming
# qc_locked = transform_circuit(qc_original)


# qc_original = qc_original.decompose(reps=3)
# qc_locked = transform_circuit(qc_original)[0]
# qc_locked = qc_locked.decompose(reps=3)

# shots = 1024
# num_runs = 100
# tvd_values = []

# # Run circuits multiple times, compute TVD
# for _ in range(num_runs):
#     job_orig = backend.run(qc_original, shots=shots)
#     result_orig = job_orig.result()
#     counts_orig = result_orig.get_counts()

#     job_lock = backend.run(qc_locked, shots=shots)
#     result_lock = job_lock.result()
#     counts_lock = result_lock.get_counts()

#     tvd = total_variation_distance_from_counts(counts_orig, counts_lock, shots)
#     tvd_values.append(tvd)

# # --- Compute statistics ---
# mean_tvd = np.mean(tvd_values)
# var_tvd = np.var(tvd_values, ddof=1)   # sample variance (ddof=1)
# std_tvd = np.std(tvd_values, ddof=1)   # standard deviation

# print(f"Mean TVD: {mean_tvd:.4f}")
# print(f"Variance of TVD: {var_tvd:.6f}")
# print(f"Standard Deviation of TVD: {std_tvd:.6f}")

# # Plot histogram of TVD values
# plt.hist(tvd_values, bins=20, color='skyblue', edgecolor='black')
# plt.axvline(mean_tvd, color='red', linestyle='--', label=f'Mean = {mean_tvd:.3f}')
# plt.title('Histogram of Total Variation Distances over 100 runs')
# plt.xlabel('Total Variation Distance')
# plt.ylabel('Frequency')
# plt.legend()
# plt.show()


In [ ]:
# import numpy as np
# import matplotlib.pyplot as plt

# # ==========================================================
# #   QASM Benchmark: TVD Comparison (Original / Locked / Restored)
# # ==========================================================

# # Use already declared qc_original, qc_locked, backend, shots, etc.

# # Suppose you have a way to "restore" the locked circuit.
# # For now, let's assume the restoration process simply reverses the transform.
# # Replace this with your actual restoration logic if different.
# qc_restored = transform_circuit(qc_locked)  # Example placeholder

# num_runs = 20  # You can keep it smaller for visualization
# tvd_obfuscation_list = []
# tvd_restoration_list = []

# for i in range(num_runs):
#     print(f"Trial {i+1}")

#     # --- Original ---
#     job_orig = backend.run(qc_original, shots=shots)
#     counts_orig = job_orig.result().get_counts()

#     # --- Locked (obfuscated) ---
#     job_lock = backend.run(qc_locked, shots=shots)
#     counts_lock = job_lock.result().get_counts()

#     # --- Restored ---
#     job_restored = backend.run(qc_restored, shots=shots)
#     counts_restored = job_restored.result().get_counts()

#     # --- Compute TVDs ---
#     tvd_obf = total_variation_distance_from_counts(counts_orig, counts_lock, shots)
#     tvd_res = total_variation_distance_from_counts(counts_orig, counts_restored, shots)

#     tvd_obfuscation_list.append(tvd_obf)
#     tvd_restoration_list.append(tvd_res)

# # --- Compute summary statistics ---
# mean_obf = np.mean(tvd_obfuscation_list)
# mean_res = np.mean(tvd_restoration_list)
# std_obf = np.std(tvd_obfuscation_list, ddof=1)
# std_res = np.std(tvd_restoration_list, ddof=1)

# print(f"\nMean TVD (Original vs Locked):   {mean_obf:.4f}")
# print(f"Mean TVD (Original vs Restored): {mean_res:.4f}")
# print(f"Std Dev (Locked): {std_obf:.6f}, Std Dev (Restored): {std_res:.6f}")

# # ==========================================================
# #   Bar Chart Visualization
# # ==========================================================

# plt.figure(figsize=(12, 6))
# indices = np.arange(num_runs)
# bar_width = 0.4

# # Bar plots for each comparison
# plt.bar(indices - bar_width/2, tvd_obfuscation_list, width=bar_width,
#         label="Original vs Locked", color='tab:orange', edgecolor='black')
# plt.bar(indices + bar_width/2, tvd_restoration_list, width=bar_width,
#         label="Original vs Restored", color='tab:blue', edgecolor='black')

# # Mean/threshold lines
# plt.axhline(y=mean_obf, color='red', linestyle=':', label=f"Mean Locked = {mean_obf:.3f}")
# plt.axhline(y=mean_res, color='green', linestyle=':', label=f"Mean Restored = {mean_res:.3f}")
# plt.axhline(y=0.01, color='gray', linestyle='--', label="Expected Restoration Threshold")

# # Axis formatting
# plt.title("QASM Benchmark: TVD Comparison (Original vs Locked vs Restored)")
# plt.xlabel("Trial")
# plt.ylabel("Total Variation Distance (TVD)")
# plt.xticks(indices)
# plt.legend()
# plt.grid(axis='y', linestyle='--', alpha=0.7)
# plt.tight_layout()
# plt.show()
